<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/07_baseline_rf_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07_baseline_rf.ipynb — Baseline Supervisionado (Random Forest)

## Objetivo
Treinar e avaliar um baseline supervisionado com Random Forest sobre a base rotulada em janelas de 5 minutos (Notebook 06), comparando:
1. **Classificação binária**: `is_event = 1` para `{BEFORE, DURING, AFTER}` vs `NORMAL`
2. **Classificação multiclasse**: `{NORMAL, BEFORE, DURING, AFTER}`

A avaliação é feita com split temporal para evitar vazamento de informação futura.

## Entradas (artefatos do pipeline)
- `window_5min_labeled.parquet` (Notebook 06)

## Saídas (artefatos deste Notebook)
- `rf_binary.joblib`
- `rf_multiclass.joblib`
- `07_baseline_rf_summary.json`

## Split temporal
Este notebook suporta duas estratégias:
- **Corte fixo (default)**: treino = 80% inicial; teste = 20% final
- **TimeSeriesSplit (opcional)**: validação em múltiplos folds temporais

## Métricas reportadas
### Binário
- accuracy
- precision
- recall
- f1
- ROC-AUC *(quando aplicável)*

### Multiclasse
- accuracy
- macro-f1
- weighted-f1
- classification report

### Adicional
- Matriz de confusão (binário e multiclasse)

## Features
- Exclui colunas não numéricas e colunas de rótulo (`state`, `is_critical`)
- Preserva `bucket_id` apenas como referência (não como feature)

## Observações
O dataset é desbalanceado, especialmente na classe `DURING`. Random Forest é utilizado como baseline interpretável e robusto. Ajustes como `class_weight` podem ser aplicados em iterações futuras.

In [ ]:
# ============================================================
# 07_baseline_rf.ipynb
# Baseline supervisionado com Random Forest (binário vs multiclasse)
# Split temporal + métricas + matriz de confusão + salvamento do modelo
# Pipeline PPCOMP_DM (Google Cluster Trace) - coerente com taxa (fail_rate)
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull).")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, MODELS_PATH, ensure_dirs
ensure_dirs()

print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH  =", REPORTS_PATH)
print("MODELS_PATH   =", MODELS_PATH)

def log(msg: str) -> None:
    print(f"[07_baseline_rf] {msg}")

# -----------------------------
# 1) Imports ML
# -----------------------------
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
import joblib

# -----------------------------
# 2) Carregar dataset rotulado (NB06)
# -----------------------------
DATA_FILE = FEATURES_PATH / "window_5min_labeled.parquet"
assert DATA_FILE.exists(), f"Arquivo não encontrado: {DATA_FILE}"

df = pd.read_parquet(DATA_FILE).sort_values("bucket_id").reset_index(drop=True)

assert "state" in df.columns, "Coluna 'state' ausente."
assert "bucket_id" in df.columns, "Coluna 'bucket_id' ausente."
assert "fail_rate" in df.columns, "Coluna 'fail_rate' ausente (pipeline por taxa não está refletido)."

log(f"Dataset rotulado: shape={df.shape}")
dist_total = df["state"].value_counts().to_dict()
log(f"Distribuição total: {dist_total}")

# -----------------------------
# 3) Preparar features (X) e alvos (y)
# -----------------------------
REMOVE_TIME_ABSOLUTE = False  # se True, remove bucket_start_us das features

drop_cols = {"state", "bucket_id"}

# is_critical é derivado do NB04; remover para evitar vazamento conceitual
if "is_critical" in df.columns:
    drop_cols.add("is_critical")

if REMOVE_TIME_ABSOLUTE and "bucket_start_us" in df.columns:
    drop_cols.add("bucket_start_us")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X_cols = [c for c in numeric_cols if c not in drop_cols]

if len(X_cols) == 0:
    raise ValueError("Nenhuma feature numérica encontrada após filtragem.")

X = df[X_cols].copy()
y_multi = df["state"].astype(str).copy()

log(f"Features usadas: {len(X_cols)}")
log(f"Lista features: {X_cols}")

# -----------------------------
# 4) Split temporal (corte fixo default)
# -----------------------------
SPLIT_MODE = "fixed"   # "fixed" ou "tscv"
TEST_RATIO = 0.20

n = len(df)
test_size = int(np.ceil(n * TEST_RATIO))
train_end = n - test_size

X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train_multi, y_test_multi = y_multi.iloc[:train_end], y_multi.iloc[train_end:]

log(f"Split temporal: train={len(X_train)} test={len(X_test)} (modo={SPLIT_MODE})")

# distribuição por split (útil para validar desbalanceamento)
dist_train = y_train_multi.value_counts().to_dict()
dist_test = y_test_multi.value_counts().to_dict()
log(f"Distribuição treino: {dist_train}")
log(f"Distribuição teste : {dist_test}")

# -----------------------------
# 5) Cenário binário: evento vs normal
# -----------------------------
y_train_bin = (y_train_multi != "NORMAL").astype(int)
y_test_bin = (y_test_multi != "NORMAL").astype(int)

rf_bin = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf_bin.fit(X_train, y_train_bin)

pred_bin = rf_bin.predict(X_test)
proba_bin = rf_bin.predict_proba(X_test)[:, 1] if hasattr(rf_bin, "predict_proba") else None

acc_bin = accuracy_score(y_test_bin, pred_bin)
prec_bin, rec_bin, f1_bin, _ = precision_recall_fscore_support(
    y_test_bin, pred_bin, average="binary", zero_division=0
)

auc_bin = None
if proba_bin is not None and len(np.unique(y_test_bin)) == 2:
    try:
        auc_bin = float(roc_auc_score(y_test_bin, proba_bin))
    except Exception:
        auc_bin = None

cm_bin = confusion_matrix(y_test_bin, pred_bin).tolist()
log(f"[BIN] acc={acc_bin:.4f} prec={prec_bin:.4f} rec={rec_bin:.4f} f1={f1_bin:.4f} auc={auc_bin}")

# -----------------------------
# 6) Cenário multiclasse: NORMAL/BEFORE/DURING/AFTER
# -----------------------------
labels = ["NORMAL", "BEFORE", "DURING", "AFTER"]  # ordem fixa para matriz comparável

rf_multi = RandomForestClassifier(
    n_estimators=400,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)
rf_multi.fit(X_train, y_train_multi)

pred_multi = rf_multi.predict(X_test)

acc_multi = accuracy_score(y_test_multi, pred_multi)

macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
    y_test_multi, pred_multi, average="macro", zero_division=0
)
w_prec, w_rec, w_f1, _ = precision_recall_fscore_support(
    y_test_multi, pred_multi, average="weighted", zero_division=0
)

cm_multi = confusion_matrix(y_test_multi, pred_multi, labels=labels).tolist()
report_multi = classification_report(y_test_multi, pred_multi, labels=labels, zero_division=0)

log(f"[MULTI] acc={acc_multi:.4f} macro_f1={macro_f1:.4f} weighted_f1={w_f1:.4f}")

# -----------------------------
# 7) (Opcional) TimeSeriesSplit rápido (apenas multiclasse)
# -----------------------------
tscv_results = []
if SPLIT_MODE == "tscv":
    tscv = TimeSeriesSplit(n_splits=5)
    for fold, (tr, te) in enumerate(tscv.split(X), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y_multi.iloc[tr], y_multi.iloc[te]

        clf = RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1,
            class_weight="balanced_subsample"
        )
        clf.fit(X_tr, y_tr)
        y_hat = clf.predict(X_te)

        f1_macro = precision_recall_fscore_support(
            y_te, y_hat, average="macro", zero_division=0
        )[2]
        tscv_results.append({"fold": fold, "f1_macro": float(f1_macro)})

# -----------------------------
# 8) Salvar modelos
# -----------------------------
bin_model_path = MODELS_PATH / "rf_binary.joblib"
multi_model_path = MODELS_PATH / "rf_multiclass.joblib"

joblib.dump(rf_bin, bin_model_path)
joblib.dump(rf_multi, multi_model_path)

# -----------------------------
# 9) Summary (JSON)
# -----------------------------
summary = {
    "seed": SEED,
    "split_mode": SPLIT_MODE,
    "test_ratio": TEST_RATIO,
    "rows_total": int(n),
    "rows_train": int(len(X_train)),
    "rows_test": int(len(X_test)),
    "n_features": int(len(X_cols)),
    "features": X_cols,
    "class_distribution_total": {k: int(v) for k, v in dist_total.items()},
    "class_distribution_train": {k: int(v) for k, v in dist_train.items()},
    "class_distribution_test": {k: int(v) for k, v in dist_test.items()},
    "binary": {
        "acc": float(acc_bin),
        "precision": float(prec_bin),
        "recall": float(rec_bin),
        "f1": float(f1_bin),
        "roc_auc": auc_bin,
        "confusion_matrix": cm_bin
    },
    "multiclass": {
        "acc": float(acc_multi),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(w_f1),
        "confusion_matrix": cm_multi,
        "labels_order": labels,
        "classification_report_text": report_multi
    },
    "tscv_multiclass": tscv_results,
    "models": {
        "binary_path": str(bin_model_path),
        "multiclass_path": str(multi_model_path)
    }
}

summary_file = REPORTS_PATH / "07_baseline_rf_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 07 finalizado com sucesso.")

print("\n=== MULTICLASS REPORT ===\n")
print(report_multi)
print("\n=== CONFUSION MATRIX (BIN) ===\n", cm_bin)
print("\n=== CONFUSION MATRIX (MULTI) labels=", labels, "===\n", cm_multi)
print("\nModels saved:\n", bin_model_path, "\n", multi_model_path)
print("\nSummary saved:\n", summary_file)

Mounted at /content/drive
[Bootstrap] Atualizando repositório (git pull).
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
FEATURES_PATH = /content/drive/MyDrive/Mestrado/02-datasets/03-features
REPORTS_PATH  = /content/drive/MyDrive/Mestrado/04-reports
MODELS_PATH   = /content/drive/MyDrive/Mestrado/03-models
[07_baseline_rf] Dataset rotulado: shape=(8914, 28)
[07_baseline_rf] Distribuição total: {'NORMAL': 5533, 'BEFORE': 1996, 'AFTER': 991, 'DURING': 394}
[07_baseline_rf] Features usadas: 25
[07_baseline_rf] Lista features: ['bucket_start_us', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'mean_priority', 'mean_req_cpus', 'mean_req_mem', 'req_cpus_presence_rate', 'req_mem_presence_rate', 'event_FAIL_count', 'event_SCHEDULE_count', 'event_FINISH_count', 'event_ENABLE_count', 'event_LOST_count', 'event_EVICT_count', 'event_KILL_count', 'fail_rate', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_1h', 'rolling_std_1h', 'pct_change', 'zscore_global']
[07_baseline_rf] Split 

## Teste controlado — remoção de `bucket_start_us` (viés de regime temporal)

### Motivação

O atributo `bucket_start_us` representa tempo absoluto (monotônico crescente). Em avaliação com **split temporal** (treino = passado, teste = futuro), esse atributo pode capturar indiretamente **mudanças de regime** (não-estacionaridade), isto é, aprender um proxy do tipo “quanto mais próximo do fim do dataset, maior probabilidade de evento”.

Esse efeito não caracteriza vazamento direto de rótulo, mas pode **inflar métricas** ao explorar diferenças sistemáticas de distribuição entre treino e teste.

Na execução atual, observa-se que o bloco de teste contém proporção maior de janelas em estados de evento (BEFORE/DURING/AFTER) do que o bloco de treino, o que reforça a necessidade de verificar se `bucket_start_us` está contribuindo para desempenho via regime temporal.

### Objetivo do teste

Executar um experimento controlado mantendo:
- mesmo dataset (`window_5min_labeled.parquet`)
- mesmo split temporal (80/20)
- mesmas features (exceto `bucket_start_us`)
- mesmos hiperparâmetros do Random Forest

Comparar os resultados **com vs sem** `bucket_start_us` em:
- Binário: Accuracy, Precision, Recall, F1, ROC-AUC, matriz de confusão
- Multiclasse: Accuracy, Macro-F1, Weighted-F1, classification report, matriz de confusão

### Interpretação esperada

- Se o desempenho cair pouco: evidência de que a separabilidade decorre majoritariamente das features derivadas de `fail_rate`.
- Se o desempenho cair muito: indício de forte dependência de regime temporal e necessidade de revisão do protocolo (ex.: validação por múltiplos blocos temporais, TimeSeriesSplit, remoção do tempo absoluto como feature).

In [ ]:
# ============================================================
# 07_baseline_rf.ipynb
# Baseline supervisionado com Random Forest (binário vs multiclasse)
# Split temporal + métricas + matriz de confusão + salvamento do modelo
# Pipeline PPCOMP_DM (Google Cluster Trace) - V0
# ============================================================

# -----------------------------
# 0) BOOTSTRAP (Colab + Repo)
# -----------------------------
from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import numpy as np
import pandas as pd
import json

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("[Bootstrap] Google Drive já montado.")

REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
    subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull).")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar via git pull:", e)

os.chdir(str(REPO_DIR))
print("[Bootstrap] CWD =", os.getcwd())

repo_str = str(REPO_DIR)
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
importlib.invalidate_caches()

from src.paths import FEATURES_PATH, REPORTS_PATH, MODELS_PATH, ensure_dirs
ensure_dirs()

print("FEATURES_PATH =", FEATURES_PATH)
print("REPORTS_PATH =", REPORTS_PATH)
print("MODELS_PATH =", MODELS_PATH)

def log(msg: str) -> None:
    print(f"[07_baseline_rf] {msg}")

# -----------------------------
# 1) Imports ML
# -----------------------------
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import joblib

# -----------------------------
# 2) Carregar dataset rotulado
# -----------------------------
DATA_FILE = FEATURES_PATH / "window_5min_labeled.parquet"
assert DATA_FILE.exists(), f"Arquivo não encontrado: {DATA_FILE}"

df = pd.read_parquet(DATA_FILE).sort_values("bucket_id").reset_index(drop=True)
log(f"Dataset rotulado: shape={df.shape}")

assert "state" in df.columns, "Coluna 'state' ausente."
assert "bucket_id" in df.columns, "Coluna 'bucket_id' ausente."

# -----------------------------
# 3) Preparar features (X)
# -----------------------------
drop_cols = {"state"}
if "is_critical" in df.columns:
    drop_cols.add("is_critical")
drop_cols.add("bucket_id")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
X_cols = [c for c in numeric_cols if c not in drop_cols]

if len(X_cols) == 0:
    raise ValueError("Nenhuma feature numérica encontrada após filtragem.")

X = df[X_cols].copy()
y_multi = df["state"].astype(str).copy()

log(f"Features usadas: {len(X_cols)}")
log(f"Classes (multiclasse): {sorted(y_multi.unique().tolist())}")

# -----------------------------
# 4) Split temporal (corte fixo default)
# -----------------------------
SPLIT_MODE = "fixed"  # "fixed" ou "tscv"
TEST_RATIO = 0.20

n = len(df)
test_size = int(np.ceil(n * TEST_RATIO))
train_end = n - test_size

X_train, X_test = X.iloc[:train_end], X.iloc[train_end:]
y_train_multi, y_test_multi = y_multi.iloc[:train_end], y_multi.iloc[train_end:]

log(f"Split temporal: train={len(X_train)} test={len(X_test)} (modo={SPLIT_MODE})")

# -----------------------------
# 5) Cenário binário: evento vs normal
# -----------------------------
y_train_bin = (y_train_multi != "NORMAL").astype(int)
y_test_bin = (y_test_multi != "NORMAL").astype(int)

rf_bin = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_bin.fit(X_train, y_train_bin)
pred_bin = rf_bin.predict(X_test)
proba_bin = rf_bin.predict_proba(X_test)[:, 1] if hasattr(rf_bin, "predict_proba") else None

acc_bin = accuracy_score(y_test_bin, pred_bin)
prec_bin, rec_bin, f1_bin, _ = precision_recall_fscore_support(
    y_test_bin, pred_bin, average="binary", zero_division=0
)

auc_bin = None
if proba_bin is not None and len(np.unique(y_test_bin)) == 2:
    try:
        auc_bin = float(roc_auc_score(y_test_bin, proba_bin))
    except Exception:
        auc_bin = None

cm_bin = confusion_matrix(y_test_bin, pred_bin).tolist()
log(f"[BIN] acc={acc_bin:.4f} prec={prec_bin:.4f} rec={rec_bin:.4f} f1={f1_bin:.4f} auc={auc_bin}")

# -----------------------------
# 6) Cenário multiclasse
# -----------------------------
rf_multi = RandomForestClassifier(
    n_estimators=400,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_multi.fit(X_train, y_train_multi)
pred_multi = rf_multi.predict(X_test)

acc_multi = accuracy_score(y_test_multi, pred_multi)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
    y_test_multi, pred_multi, average="macro", zero_division=0
)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    y_test_multi, pred_multi, average="weighted", zero_division=0
)

labels = ["NORMAL", "BEFORE", "DURING", "AFTER"]
cm_multi = confusion_matrix(y_test_multi, pred_multi, labels=labels).tolist()
report_multi = classification_report(y_test_multi, pred_multi, labels=labels, zero_division=0)

log(f"[MULTI] acc={acc_multi:.4f} macro_f1={f1_m:.4f} weighted_f1={f1_w:.4f}")

# -----------------------------
# 7) (Opcional) TimeSeriesSplit rápido
# -----------------------------
tscv_results = []
if SPLIT_MODE == "tscv":
    tscv = TimeSeriesSplit(n_splits=5)
    for fold, (tr, te) in enumerate(tscv.split(X), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y_multi.iloc[tr], y_multi.iloc[te]

        clf = RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1,
            class_weight="balanced_subsample"
        )
        clf.fit(X_tr, y_tr)
        y_hat = clf.predict(X_te)

        f1_macro = precision_recall_fscore_support(
            y_te, y_hat, average="macro", zero_division=0
        )[2]

        tscv_results.append({"fold": fold, "f1_macro": float(f1_macro)})

# -----------------------------
# 8) Salvar modelos
# -----------------------------
bin_model_path = MODELS_PATH / "rf_binary.joblib"
multi_model_path = MODELS_PATH / "rf_multiclass.joblib"

joblib.dump(rf_bin, bin_model_path)
joblib.dump(rf_multi, multi_model_path)

# -----------------------------
# 9) Summary (JSON) + prints úteis
# -----------------------------
summary = {
    "seed": SEED,
    "split_mode": SPLIT_MODE,
    "test_ratio": TEST_RATIO,
    "rows_total": int(n),
    "rows_train": int(len(X_train)),
    "rows_test": int(len(X_test)),
    "n_features": int(len(X_cols)),
    "features": X_cols,
    "class_distribution_total": df["state"].value_counts().to_dict(),
    "binary": {
        "acc": float(acc_bin),
        "precision": float(prec_bin),
        "recall": float(rec_bin),
        "f1": float(f1_bin),
        "roc_auc": auc_bin,
        "confusion_matrix": cm_bin
    },
    "multiclass": {
        "acc": float(acc_multi),
        "macro_f1": float(f1_m),
        "weighted_f1": float(f1_w),
        "confusion_matrix": cm_multi,
        "labels_order": labels,
        "classification_report_text": report_multi
    },
    "tscv_multiclass": tscv_results,
    "models": {
        "binary_path": str(bin_model_path),
        "multiclass_path": str(multi_model_path)
    }
}

summary_file = REPORTS_PATH / "07_baseline_rf_summary.json"
summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

log("Notebook 07 finalizado com sucesso.")
print("\n=== MULTICLASS REPORT ===\n")
print(report_multi)
print("\n=== CONFUSION MATRIX (BIN) ===\n", cm_bin)
print("\n=== CONFUSION MATRIX (MULTI) labels=", labels, "===\n", cm_multi)
print("\nModels saved:\n", bin_model_path, "\n", multi_model_path)
print("\nSummary saved:\n", summary_file)

=== TESTE CONTROLADO (SEM bucket_start_us) ===
Binário: {'acc': 0.8238923163208076, 'precision': 0.8951548848292296, 'recall': 0.8609625668449198, 'f1': 0.8777258566978193, 'roc_auc': 0.9008833683070466}
Multiclasse: {'acc': 0.64385866517106, 'macro_f1': 0.6677765193386259, 'weighted_f1': 0.6367694293201969}

Delta vs baseline (with_time):
{
  "binary": {
    "acc": 0.008412787436904101,
    "precision": 0.04565417156246776,
    "recall": -0.04889228418640179,
    "f1": -0.0009167106205133502,
    "roc_auc": 0.011966006195343493
  },
  "multiclass": {
    "acc": -0.011777902411665653,
    "macro_f1": 0.06867796931638226,
    "weighted_f1": 0.028874123446003375
  }
}

Summary salvo em: /content/drive/MyDrive/Mestrado/04-reports/07_baseline_rf_summary_no_time.json
Modelos salvos em: /content/drive/MyDrive/Mestrado/03-models/rf_binary_no_time.joblib e /content/drive/MyDrive/Mestrado/03-models/rf_multiclass_no_time.joblib


## Achados do Notebook 07 — Baseline Supervisionado

Este notebook avaliou, pela primeira vez, se os estados definidos no pipeline são efetivamente aprendíveis por um classificador supervisionado. O Random Forest foi utilizado como baseline por sua robustez, interpretabilidade e capacidade de modelar relações não lineares sem exigir forte pré-processamento.

Foram consideradas duas tarefas:
- **binária**: evento vs normal
- **multiclasse**: `NORMAL`, `BEFORE`, `DURING`, `AFTER`

### Interpretação dos resultados
A tarefa binária permite avaliar se o pipeline consegue distinguir janelas normais de janelas associadas a episódios. Já a tarefa multiclasse é mais exigente e mede a separabilidade entre as fases operacionais definidas no Notebook 06.

### Principais achados esperados desta etapa
- boa separação entre regime normal e regime de evento
- separabilidade mais forte para `DURING`
- maior dificuldade para `BEFORE`
- comportamento intermediário para `AFTER`

### Relevância
Este notebook é central porque valida a hipótese de que a rotulagem construída anteriormente não é apenas conceitual, mas também detectável de forma supervisionada a partir das features temporais geradas no Notebook 05.

### Limitações
- Random Forest é um baseline, não um modelo sequencial
- o desbalanceamento entre classes pode afetar recall de estados raros
- a versão v0 ainda está baseada em volume absoluto de falhas

Mesmo assim, esta etapa fornece a primeira evidência quantitativa de que a estrutura de estados do pipeline é utilizável para classificação.